## Concept primer: confidence intervals + hypothesis tests

**Standard error (SE)** = how much a sample statistic would vary from sample to sample.
- Mean: `SE = s / sqrt(n)` (`s` = sample std, `ddof=1`).
- Proportion: `SE = sqrt(p(1-p)/n)`.

**Central limit theorem:** sample means are roughly normal for large n, *even if the data isn't* (e.g. skewed transaction amounts). That's why these formulas work.

**Confidence interval** = `estimate ± critical value × SE`.
- 95% → critical value ≈ 1.96 (normal). For a mean with unknown σ, use the **t** distribution with `n-1` df (wider for small n): `stats.t.ppf(0.975, n-1)`.
- Meaning: if you repeated the sampling many times, 95% of intervals built this way would contain the true value. **Not** "95% chance the true value is in this interval".

**Hypothesis test**
- H0 = "no difference". H1 = "there is a difference".
- **p-value = probability of seeing data at least this extreme *if H0 were true*.** It is **not** P(H0 is true).
- p < α (usually 0.05) → reject H0. p ≥ α → "not enough evidence", **not** "proved no difference".
- Type I error = false positive (rate α). Type II = false negative. Power = 1 − P(type II).

**Two-proportion z-test** (A/B test on conversion rates):
```
p_pool = (x_a + x_b) / (n_a + n_b)              # pooled, because H0 says the rates are equal
SE     = sqrt(p_pool(1-p_pool)(1/n_a + 1/n_b))
z      = (p_b - p_a) / SE
p      = 2 * P(Z > |z|)  = 2 * stats.norm.sf(abs(z))     # two-sided
```
For the **CI of the difference**, use the **unpooled** SE: `sqrt(p_a(1-p_a)/n_a + p_b(1-p_b)/n_b)`.

**Tiny example:** 50/1000 vs 70/1000 → p_pool = 0.06, SE ≈ 0.0106, z ≈ 1.88, p ≈ 0.06 → not significant at 5%.

## Problem: did the new loan banner work?

The mobile app team ran an A/B test on a personal-loan banner.
- **A (old banner):** shown to 4,000 customers, **200** applied.
- **B (new banner):** shown to 4,000 customers, **248** applied.

Write three functions with `numpy` and `scipy.stats`. Use scipy only for `.ppf` / `.sf` / `.cdf`, not the ready-made tests. The tests check you against `scipy` / `statsmodels`.

| Function | Returns | Notes |
|---|---|---|
| `mean_ci(x, conf=0.95)` | `(mean, lo, hi)` | t-based CI for a mean |
| `two_prop_ztest(x_a, n_a, x_b, n_b)` | `(z, p_value)` | pooled SE, two-sided, `z` > 0 when B > A |
| `diff_ci(x_a, n_a, x_b, n_b, conf=0.95)` | `(diff, lo, hi)` | for `p_b - p_a`, unpooled SE |

**`mean_ci` is separate from the banner.** It takes any list of numbers (e.g. spend amounts) and `n = len(x)`. Worked example, `x = [12, 15.5, 9, 22, 18.5]`:

| Step | What | Value |
|---|---|---|
| 1 | `n = len(x)` | 5 |
| 2 | mean | 15.4 |
| 3 | `s` = sample std, `ddof=1` | 5.14 |
| 4 | `SE = s / sqrt(n)` | 2.30 |
| 5 | `t_crit = stats.t.ppf(1 - (1 - conf) / 2, n - 1)` (= 0.975 for 95%: 2.5% in each tail) | 2.78 |
| 6 | `mean ± t_crit * SE` | **(9.02, 21.78)** |

So `mean_ci([12, 15.5, 9, 22, 18.5])` → `(15.4, 9.02, 21.78)`.

**The banner functions** work on rates: `p_a = 200 / 4000 = 0.05`. Same idea (estimate ± critical value × SE), proportion formulas from the primer.
Example: `two_prop_ztest(50, 1000, 70, 1000)` → `(1.883..., 0.0597...)`.

Then answer the business question in "Explain it".

**Time target: 25 minutes.** Note your start time.

In [ ]:
import numpy as np
from scipy import stats

def mean_ci(x, conf=0.95):
    # your code here
    pass

def two_prop_ztest(x_a, n_a, x_b, n_b):
    # your code here
    pass

def diff_ci(x_a, n_a, x_b, n_b, conf=0.95):
    # your code here
    pass


In [ ]:
# Tests: run this cell. All asserts must pass silently.
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep
close = lambda a, b: np.allclose(a, b, atol=1e-6)

rng = np.random.default_rng(0)
amounts = rng.lognormal(3.3, 1.1, size=500)          # skewed spend amounts, like the bank DB
small = np.array([12.0, 15.5, 9.0, 22.0, 18.5])      # n = 5: t is much wider than z here

# 1. mean_ci vs scipy
for x in (amounts, small):
    m, lo, hi = mean_ci(x)
    ref = stats.t.interval(0.95, len(x) - 1, loc=x.mean(), scale=stats.sem(x))
    assert close(m, x.mean()) and close((lo, hi), ref), (lo, hi, ref)
assert close(mean_ci(small, 0.99)[1:], stats.t.interval(0.99, 4, loc=small.mean(), scale=stats.sem(small)))
assert close(mean_ci([5, 5, 5, 5])[1:], (5, 5))      # no variation -> zero-width CI

# 2. two_prop_ztest vs statsmodels
for args in [(200, 4000, 248, 4000), (50, 1000, 70, 1000), (30, 500, 45, 900)]:
    x_a, n_a, x_b, n_b = args
    ref_z, ref_p = proportions_ztest([x_b, x_a], [n_b, n_a])
    z, p = two_prop_ztest(*args)
    assert close(z, ref_z) and close(p, ref_p), (args, z, p, ref_z, ref_p)
z, p = two_prop_ztest(248, 4000, 200, 4000)
assert z < 0                                          # B worse than A -> negative z
z, p = two_prop_ztest(100, 1000, 100, 1000)
assert close(z, 0) and close(p, 1)                    # identical rates

# 3. diff_ci vs statsmodels (Wald = unpooled)
for args in [(200, 4000, 248, 4000), (30, 500, 45, 900)]:
    x_a, n_a, x_b, n_b = args
    d, lo, hi = diff_ci(*args)
    ref = confint_proportions_2indep(x_b, n_b, x_a, n_a, method="wald", compare="diff")
    assert close(d, x_b / n_b - x_a / n_a) and close((lo, hi), ref), (lo, hi, ref)

print("all visible tests passed")

In [ ]:
# Run once your tests pass: the actual result
z, p = two_prop_ztest(200, 4000, 248, 4000)
d, lo, hi = diff_ci(200, 4000, 248, 4000)
print(f"A = {200/4000:.2%}, B = {248/4000:.2%}")
print(f"z = {z:.2f}, p = {p:.4f}")
print(f"B - A = {d:.2%}  (95% CI {lo:.2%} to {hi:.2%})")

## Explain it

1. Complexity of each function (time AND space, biggest term only).
2. Why pooled SE in the z-test but unpooled SE in the CI?
3. Write the result as **one sentence to the product manager**: finding → number → recommendation. Then one caveat.
4. Define the p-value you got **in words**, using this test's numbers.
5. The team says "we checked the result every day and stopped as soon as p < 0.05". What's wrong with that?

_your answers here_